## IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv("training.1600000.processed.noemoticon.csv",
                 encoding='latin-1',
                 names=columns)

df = df[['target', 'text']]
df.head()

,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [3]:
# CHECK BALANCE
print(df['target'].value_counts())


target
0    800000
4    800000
Name: count, dtype: int64


In [4]:
# CLEAN TEXT
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)


In [5]:
# TOKENIZATION
vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])

sequences = tokenizer.texts_to_sequences(df['text'])
X = pad_sequences(sequences, maxlen=max_length, padding='post')

y = df['target'].values


In [6]:
# SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
# BUILD MODEL
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 64)           640000    
                                                                 
 lstm (LSTM)                 (None, 64)                33024     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 673089 (2.57 MB)
Trainable params: 673089 (2.57 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
# TRAIN MODEL
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/10
40000/40000 [==============================] - 3470s 87ms/step - loss: -1231.7108 - accuracy: 0.0000e+00 - val_loss: -2474.6538 - val_accuracy: 0.0000e+00
Epoch 2/10
 2152/40000 [>.............................] - ETA: 58:48 - loss: -2514.8921 - accuracy: 0.0000e+00

In [ ]:
# EVALUATE
loss, accuracy = model.evaluate(X_test, y_test)
print('Accuracy:', accuracy)


In [ ]:
# SAVE MODEL
model.save('sentiment_model.keras')


In [ ]:
# SAVE TOKENIZER
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)


In [ ]:
# TEST FUNCTION
def predict(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length, padding='post')
    pred = model.predict(padded)[0][0]
    return pred

print(predict('I love this product'))
print(predict('I hate this'))
